In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from urllib.parse import urlencode
from urllib.request import urlopen
from urllib.request import Request

org = pd.read_excel(
    "../data/Organization.xlsx"
)

print("Shape:", org.shape)

location_keywords = [
    "lat",
    "lon",
    "address",
    "city",
    "zip",
    "county",
    "geoid",
    "block",
    "tract"
]

location_columns = []

for column in org.columns:
    column_lower = column.lower()

    for keyword in location_keywords:
        if keyword in column_lower:
            location_columns.append(column)
            break

print("\nPossible location columns:")
print(location_columns)

display(
    org[
        location_columns
    ].head(10)
)

print("\nMissing values:")
print(
    org[
        location_columns
    ].isna().sum()
)

Shape: (1202, 68)

Possible location columns:
['City', 'Zip', 'MailCity', 'MailZip', 'County', 'block_geoid', 'is_zip_centroid']


,City,Zip,MailCity,MailZip,County,block_geoid,is_zip_centroid
0,Rochester,55904.0,Rochester,55904,Olmsted,2.710900e+14,0.0
1,Brooklyn Park,55428.0,NaN,NaN,Hennepin-Suburbs,2.705303e+14,0.0
2,East Grand Forks,56721.0,East Grand Forks,56721,Polk,2.711902e+14,0.0
3,Duluth,55808.0,Duluth,55808,Saint Louis,2.713700e+14,0.0
4,Fargo,58102.0,Fargo,58102,Clay,3.801700e+14,0.0
5,Anoka,55303.0,Anoka,55303,Hennepin-Suburbs,2.700305e+14,0.0
6,Albany,56307.0,Albany,56307,Stearns,2.714501e+14,0.0
7,Minneapolis,55409.0,Minneapolis,55409,Hennepin-Minneapolis,2.705311e+14,0.0
8,Rice,56367.0,Rice,56367,Benton,2.700902e+14,0.0
9,Annandale,55302.0,Annandale,55302,Wright,2.717110e+14,0.0



Missing values:
City               141
Zip                148
MailCity           293
MailZip            293
County              62
block_geoid        150
is_zip_centroid    150
dtype: int64


In [2]:
activity = pd.read_csv(
    "../data/processed/site_month_clean.csv",
    low_memory=False
)

print("Activity shape:", activity.shape)
print("Unique site IDs:", activity["_id_site"].nunique())

Activity shape: (28154, 65)
Unique site IDs: 633


In [3]:
activity_2024 = activity[
    (activity["_Year"] == 2024)
    & (activity["County"].notna())
    & (activity["stable_type"] != "Meal Program")
].copy()

print("2024 rows:", len(activity_2024))
print("2024 active sites:", activity_2024["_id_site"].nunique())

2024 rows: 5619
2024 active sites: 484


In [4]:
site_supply = (
    activity_2024
    .groupby("_id_site", as_index=False)
    .agg(
        annual_visits=("HouseholdsReg", "sum"),
        annual_individuals=("IndividualsReg", "sum"),
        annual_pounds=("PoundsReg", "sum"),
        activity_county=("County", "first"),
        stable_type=("stable_type", "first"),
        food_bank=("FoodBank", "first")
    )
)

print("Supply sites:", len(site_supply))

display(
    site_supply.head()
)

Supply sites: 484


,_id_site,annual_visits,annual_individuals,annual_pounds,activity_county,stable_type,food_bank
0,1066.0,26353.0,74688.0,1317093.0,Hennepin,Food Shelf,Second Harvest Heartland
1,1067.0,438.0,945.0,47171.0,Stearns,Food Shelf,Second Harvest Heartland
2,1069.0,4008.0,5768.0,282968.0,Hennepin,Food Shelf,The Food Group
3,1070.0,527.0,1233.0,44795.0,Benton,Food Shelf,Second Harvest Heartland
4,1071.0,3604.0,8445.0,376090.0,Wright,Food Shelf,Second Harvest Heartland


In [5]:
site_supply["_id_site"] = pd.to_numeric(site_supply["_id_site"], errors="coerce").fillna(0).astype("int64")
org["__ID"] = pd.to_numeric(org["__ID"], errors="coerce").fillna(0).astype("int64")


In [6]:
print(
    "Organization IDs unique:",
    org["__ID"].is_unique
)

print(
    "Duplicate organization IDs:",
    org["__ID"].duplicated().sum()
)

Organization IDs unique: False
Duplicate organization IDs: 5


In [7]:
print(
    "Supply IDs unique:",
    site_supply["_id_site"].is_unique
)

print(
    "Missing supply IDs:",
    site_supply["_id_site"].isna().sum()
)

print(
    "\nMissing numeric organization IDs:",
    org["__ID"].isna().sum()
)

numeric_org = org[
    org["__ID"].notna()
].copy()

duplicate_org_ids = numeric_org[
    numeric_org["__ID"].duplicated(
        keep=False
    )
].copy()

print(
    "Duplicate NON-MISSING organization IDs:",
    duplicate_org_ids["__ID"].nunique()
)

print(
    "Rows belonging to duplicate IDs:",
    len(duplicate_org_ids)
)

display(
    duplicate_org_ids[
        [
            "__ID",
            "City",
            "Zip",
            "County",
            "block_geoid",
            "is_zip_centroid"
        ]
    ]
    .sort_values("__ID")
    .head(30)
)

Supply IDs unique: True
Missing supply IDs: 0

Missing numeric organization IDs: 0
Duplicate NON-MISSING organization IDs: 5
Rows belonging to duplicate IDs: 10


,__ID,City,Zip,County,block_geoid,is_zip_centroid
557,14694,Minneapolis,55405.0,Hennepin-Minneapolis,2.705313e+14,0.0
589,14694,St. Paul,55118.0,Dakota,2.703706e+14,0.0
566,14696,Burnsville,55337.0,Dakota,2.703706e+14,0.0
590,14696,Minneapolis,55418.0,Hennepin-Minneapolis,2.705310e+14,0.0
569,14699,NaN,NaN,NaN,NaN,NaN
591,14699,Austin,55912.0,Mower,2.709900e+14,0.0
570,14703,NaN,NaN,Lake,NaN,NaN
592,14703,Brooklyn Center,55430.0,Hennepin-Suburbs,2.705302e+14,0.0
574,14710,NaN,NaN,St. Louis,NaN,NaN
595,14710,Minneapolis,55407.0,Hennepin-Minneapolis,2.705311e+14,0.0


In [8]:
duplicate_ids = duplicate_org_ids["__ID"].unique()

display(
    site_supply[
        site_supply["_id_site"].isin(duplicate_ids)
    ][
        [
            "_id_site",
            "activity_county",
            "annual_visits",
            "stable_type",
            "food_bank"
        ]
    ]
    .sort_values("_id_site")
)

,_id_site,activity_county,annual_visits,stable_type,food_bank
385,14694,Dakota,2773.0,Food Shelf / Outreach,Second Harvest Heartland
386,14696,Hennepin,883.0,Food Shelf,The Food Group
387,14699,Mower,4670.0,Mobile / Outreach,Channel One Regional Food Bank*
388,14703,Hennepin,0.0,Food Shelf,The Food Group
392,14710,Hennepin,19620.0,Food Shelf,The Food Group


In [9]:
org_check = duplicate_org_ids.copy()

org_check["org_county_clean"] = (
    org_check["County"]
    .replace({
        "Hennepin-Minneapolis": "Hennepin",
        "Hennepin-Suburbs": "Hennepin",
        "Ramsey-Saint Paul": "Ramsey",
        "Ramsey-Suburbs": "Ramsey",
        "Saint Louis": "St. Louis",
        "Lac Qui Parle": "Lac qui Parle"
    })
)

In [10]:
duplicate_check = site_supply[
    site_supply["_id_site"].isin(duplicate_ids)
][
    [
        "_id_site",
        "activity_county",
        "annual_visits"
    ]
].merge(
    org_check[
        [
            "__ID",
            "City",
            "Zip",
            "County",
            "org_county_clean",
            "block_geoid",
            "is_zip_centroid"
        ]
    ],
    left_on="_id_site",
    right_on="__ID",
    how="left"
)

duplicate_check["county_match"] = (
    duplicate_check["activity_county"]
    == duplicate_check["org_county_clean"]
)

display(
    duplicate_check[
        [
            "_id_site",
            "activity_county",
            "City",
            "Zip",
            "County",
            "org_county_clean",
            "block_geoid",
            "county_match"
        ]
    ]
    .sort_values(["_id_site", "county_match"])
)

,_id_site,activity_county,City,Zip,County,org_county_clean,block_geoid,county_match
0,14694,Dakota,Minneapolis,55405.0,Hennepin-Minneapolis,Hennepin,2.705313e+14,False
1,14694,Dakota,St. Paul,55118.0,Dakota,Dakota,2.703706e+14,True
2,14696,Hennepin,Burnsville,55337.0,Dakota,Dakota,2.703706e+14,False
3,14696,Hennepin,Minneapolis,55418.0,Hennepin-Minneapolis,Hennepin,2.705310e+14,True
4,14699,Mower,NaN,NaN,NaN,NaN,NaN,False
5,14699,Mower,Austin,55912.0,Mower,Mower,2.709900e+14,True
6,14703,Hennepin,NaN,NaN,Lake,Lake,NaN,False
7,14703,Hennepin,Brooklyn Center,55430.0,Hennepin-Suburbs,Hennepin,2.705302e+14,True
8,14710,Hennepin,NaN,NaN,St. Louis,St. Louis,NaN,False
9,14710,Hennepin,Minneapolis,55407.0,Hennepin-Minneapolis,Hennepin,2.705311e+14,True


In [11]:
org_resolved = org.copy()

org_resolved["org_county_clean"] = (
    org_resolved["County"]
    .replace({
        "Hennepin-Minneapolis": "Hennepin",
        "Hennepin-Suburbs": "Hennepin",
        "Ramsey-Saint Paul": "Ramsey",
        "Ramsey-Suburbs": "Ramsey",
        "Saint Louis": "St. Louis",
        "Lac Qui Parle": "Lac qui Parle"
    })
)

In [12]:
duplicate_id_mask = org_resolved[
    "__ID"
].duplicated(keep=False)

org_unique = org_resolved[
    ~duplicate_id_mask
].copy()

org_duplicates = org_resolved[
    duplicate_id_mask
].copy()

print(
    "Unique-ID rows:",
    len(org_unique)
)

print(
    "Duplicate-ID rows:",
    len(org_duplicates)
)

Unique-ID rows: 1192
Duplicate-ID rows: 10


In [13]:
duplicate_activity_counties = site_supply[
    site_supply["_id_site"].isin(
        duplicate_ids
    )
][
    [
        "_id_site",
        "activity_county"
    ]
].copy()

org_duplicates = org_duplicates.merge(
    duplicate_activity_counties,
    left_on="__ID",
    right_on="_id_site",
    how="left"
)

In [14]:
resolved_duplicates = org_duplicates[
    org_duplicates["org_county_clean"]
    == org_duplicates["activity_county"]
].copy()

print(
    "Resolved duplicate IDs:",
    resolved_duplicates["__ID"].nunique()
)

display(
    resolved_duplicates[
        [
            "__ID",
            "activity_county",
            "City",
            "Zip",
            "County",
            "block_geoid"
        ]
    ]
)

Resolved duplicate IDs: 5


,__ID,activity_county,City,Zip,County,block_geoid
5,14694,Dakota,St. Paul,55118.0,Dakota,2.703706e+14
6,14696,Hennepin,Minneapolis,55418.0,Hennepin-Minneapolis,2.705310e+14
7,14699,Mower,Austin,55912.0,Mower,2.709900e+14
8,14703,Hennepin,Brooklyn Center,55430.0,Hennepin-Suburbs,2.705302e+14
9,14710,Hennepin,Minneapolis,55407.0,Hennepin-Minneapolis,2.705311e+14


In [15]:
resolved_duplicates = resolved_duplicates.drop(
    columns=[
        "_id_site",
        "activity_county"
    ]
)

org_for_merge = pd.concat(
    [
        org_unique,
        resolved_duplicates
    ],
    ignore_index=True
)

print(
    "Organization IDs unique:",
    org_for_merge["__ID"].is_unique
)

print(
    "Rows:",
    len(org_for_merge)
)

Organization IDs unique: True
Rows: 1197


In [16]:
site_locations = site_supply.merge(
    org_for_merge[
        [
            "__ID",
            "City",
            "Zip",
            "County",
            "org_county_clean",
            "block_geoid",
            "is_zip_centroid"
        ]
    ],
    left_on="_id_site",
    right_on="__ID",
    how="left",
    validate="one_to_one"
)

print(
    "Supply sites:",
    len(site_supply)
)

print(
    "Merged rows:",
    len(site_locations)
)

print(
    "\nMissing organization match:",
    site_locations["__ID"].isna().sum()
)

print(
    "Missing block GEOID:",
    site_locations["block_geoid"].isna().sum()
)

print(
    "Missing ZIP:",
    site_locations["Zip"].isna().sum()
)

Supply sites: 484
Merged rows: 484

Missing organization match: 0
Missing block GEOID: 11
Missing ZIP: 29


In [17]:
site_locations["block_geoid_text"] = (
    pd.to_numeric(
        site_locations["block_geoid"],
        errors="coerce"
    )
    .round()
    .astype("Int64")
    .astype("string")
)

site_locations["state_fips"] = (
    site_locations["block_geoid_text"]
    .str[:2]
)

print(
    site_locations[
        "state_fips"
    ].value_counts(
        dropna=False
    )
)

state_fips
27      473
<NA>     11
Name: count, dtype: Int64


In [18]:
missing_geo = site_locations[
    site_locations["block_geoid"].isna()
].copy()

display(
    missing_geo[
        [
            "_id_site",
            "activity_county",
            "City",
            "Zip",
            "County",
            "annual_visits",
            "annual_individuals",
            "annual_pounds",
            "stable_type",
            "food_bank",
            "is_zip_centroid"
        ]
    ]
    .sort_values(
        "annual_visits",
        ascending=False
    )
)

,_id_site,activity_county,City,Zip,County,annual_visits,annual_individuals,annual_pounds,stable_type,food_bank,is_zip_centroid
324,14468,Ramsey,NaN,NaN,Ramsey-Suburbs,1362.0,3060.0,53667.0,Food Shelf / Outreach,Second Harvest Heartland,NaN
164,1284,St. Louis,NaN,NaN,St. Louis,640.0,1323.0,31972.0,Food Shelf,Second Harvest Northland,NaN
151,1266,Lake,NaN,NaN,Lake,473.0,1095.0,28373.0,Food Shelf,Second Harvest Northland,NaN
406,14763,Koochiching,International Falls,56649.0,Koochiching,421.0,379.0,1491.0,Food Shelf,Second Harvest Northland,NaN
423,14817,Hennepin,NaN,NaN,Hennepin-Suburbs,219.0,219.0,12045.0,Food Shelf,Second Harvest Heartland,NaN
285,14169,Hennepin,NaN,NaN,Hennepin-Suburbs,0.0,0.0,0.0,Food Shelf,Second Harvest Heartland,NaN
421,14811,Hennepin,Minneapolis,55423.0,Hennepin-Minneapolis,0.0,0.0,0.0,Food Shelf,The Food Group,NaN
425,14912,Aitkin,Palisade,56431.0,Aitkin,0.0,0.0,0.0,Mobile / Outreach,Second Harvest Northland,NaN
450,15324,Hennepin,NaN,NaN,Hennepin-Suburbs,0.0,0.0,0.0,Food Shelf,Second Harvest Heartland,NaN
457,15342,Clay,NaN,NaN,Clay,0.0,0.0,0.0,Food Shelf,Great Plains Food Bank,NaN


In [19]:
total_visits = site_locations["annual_visits"].sum()

missing_geo_visits = missing_geo[
    "annual_visits"
].sum()

missing_geo_share = (
    missing_geo_visits
    / total_visits
) * 100

print(
    "Sites missing block GEOID:",
    len(missing_geo)
)

print(
    "Annual visits from missing-location sites:",
    missing_geo_visits
)

print(
    "Share of all annual visits:",
    missing_geo_share
)

print(
    "\nMissing-location sites with ZIP:",
    missing_geo["Zip"].notna().sum()
)

print(
    "Missing-location sites with City:",
    missing_geo["City"].notna().sum()
)

Sites missing block GEOID: 11
Annual visits from missing-location sites: 3115.0
Share of all annual visits: 0.10985108565669262

Missing-location sites with ZIP: 3
Missing-location sites with City: 3


In [20]:
print(
    "Sites with zero annual visits:",
    (site_locations["annual_visits"] == 0).sum()
)

print(
    "Missing-location sites with zero annual visits:",
    (missing_geo["annual_visits"] == 0).sum()
)

Sites with zero annual visits: 12
Missing-location sites with zero annual visits: 6


In [21]:
supply_keywords = [
    "hour",
    "day",
    "open",
    "capacity",
    "service",
    "distribution",
    "schedule",
    "pound",
    "volume",
    "client",
    "household"
]

possible_supply_columns = []

for column in org.columns:

    column_lower = column.lower()

    for keyword in supply_keywords:

        if keyword in column_lower:
            possible_supply_columns.append(
                column
            )
            break

print(
    "Possible supply columns:"
)

print(
    possible_supply_columns
)

Possible supply columns:
['zcountDaysOperations', 'isDistributionSite', 'z12MoAverage_Pounds']


In [22]:
for number, column in enumerate(
    org.columns,
    start=1
):
    print(
        number,
        column
    )

1 __ID
2 CreateTime
3 zzCreationTS
4 zzModificationTS
5 _id_type
6 City
7 State
8 Zip
9 MailCity
10 MailState
11 MailZip
12 Eligibility
13 Conditions
14 MarketSeasons
15 zcountDaysOperations
16 Region
17 SuperRegion
18 County
19 StateDistrict
20 FederalDistrict
21 DistrictSection
22 AgencyType
23 _id_subtype
24 AgencySubType
25 isFoodBank
26 isAgency
27 isDistributionSite
28 HoldsInventoryForOthers
29 isWebscmShipTo
30 WebscmShipToName
31 StateClass
32 MFS
33 MCS
34 cCountSites
35 isActive
36 ActiveInactive
37 InactiveDate
38 TaxEINVerifiedYN
39 TaxEINVerifiedDate
40 TaxEINVerifiedMethod
41 isTEFAPEligible
42 isTEFAPEligible_Alert
43 TefapStatus
44 TefapActiveDate
45 TefapInActiveDate
46 TEFAPElligibilityMeans
47 SuperShelfStatus
48 SuperShelfCertDate
49 _id_Bank
50 _id_Agency
51 _id_IfFoodBank
52 isFindHelpMap
53 FindHelpMapCategory
54 _id_findhelpMapCat
55 z12MoAverage_children
56 z12MoAverage_Adults
57 z12MoAverage_Senior
58 z12MoAverage_Indiv
59 z12MoAverage_House
60 z12MoAverage_P

In [23]:
supply_fields = org_for_merge[
    [
        "__ID",
        "zcountDaysOperations",
        "isDistributionSite",
        "z12MoAverage_Pounds"
    ]
].copy()

site_locations = site_locations.merge(
    supply_fields,
    on="__ID",
    how="left",
    validate="one_to_one"
)

print(
    "Rows after adding supply fields:",
    len(site_locations)
)

Rows after adding supply fields: 484


In [24]:
candidate_supply = [
    "zcountDaysOperations",
    "isDistributionSite",
    "z12MoAverage_Pounds"
]

for column in candidate_supply:

    print("\n", column)

    print(
        "Missing:",
        site_locations[column]
        .isna()
        .sum()
    )

    print(
        site_locations[column]
        .describe()
    )


 zcountDaysOperations
Missing: 19
count    465.000000
mean       7.006452
std        0.080149
min        7.000000
25%        7.000000
50%        7.000000
75%        7.000000
max        8.000000
Name: zcountDaysOperations, dtype: float64

 isDistributionSite
Missing: 1
count    483.0
mean       1.0
std        0.0
min        1.0
25%        1.0
50%        1.0
75%        1.0
max        1.0
Name: isDistributionSite, dtype: float64

 z12MoAverage_Pounds
Missing: 1
count       483.000000
mean      27771.958592
std       54378.648080
min           0.000000
25%        2914.000000
50%        8470.000000
75%       29579.500000
max      490057.000000
Name: z12MoAverage_Pounds, dtype: float64


In [25]:
print(
    "Operating-days value counts:"
)

print(
    site_locations[
        "zcountDaysOperations"
    ]
    .value_counts(
        dropna=False
    )
    .sort_index()
)

print(
    "\nDistribution-site value counts:"
)

print(
    site_locations[
        "isDistributionSite"
    ]
    .value_counts(
        dropna=False
    )
)

print(
    "\n12-month average pounds:"
)

print(
    site_locations[
        "z12MoAverage_Pounds"
    ]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

Operating-days value counts:
zcountDaysOperations
7.0    462
8.0      3
NaN     19
Name: count, dtype: int64

Distribution-site value counts:
isDistributionSite
1.0    483
NaN      1
Name: count, dtype: int64

12-month average pounds:
count       483.000000
mean      27771.958592
std       54378.648080
min           0.000000
10%        1507.800000
25%        2914.000000
50%        8470.000000
75%       29579.500000
90%       67529.000000
95%      131825.900000
99%      250654.740000
max      490057.000000
Name: z12MoAverage_Pounds, dtype: float64


In [26]:
print(
    "Correlation with annual visits:"
)

print(
    site_locations[
        [
            "annual_visits",
            "zcountDaysOperations",
            "z12MoAverage_Pounds"
        ]
    ]
    .corr()
)

Correlation with annual visits:
                      annual_visits  zcountDaysOperations  z12MoAverage_Pounds
annual_visits              1.000000             -0.028663             0.757871
zcountDaysOperations      -0.028663              1.000000            -0.037057
z12MoAverage_Pounds        0.757871             -0.037057             1.000000


In [27]:
bg_url = (
    "https://www2.census.gov/"
    "geo/docs/reference/cenpop2020/blkgrp/"
    "CenPop2020_Mean_BG27.txt"
)

bg = pd.read_csv(
    bg_url,
    dtype={
        "STATEFP": str,
        "COUNTYFP": str,
        "TRACTCE": str,
        "BLKGRPCE": str
    }
)

print(
    "Block groups:",
    len(bg)
)

print(
    bg.columns.tolist()
)

display(
    bg.head()
)

Block groups: 4706
['STATEFP', 'COUNTYFP', 'TRACTCE', 'BLKGRPCE', 'POPULATION', 'LATITUDE', 'LONGITUDE']


,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,POPULATION,LATITUDE,LONGITUDE
0,27,001,770100,1,813,46.850330,-93.275389
1,27,001,770100,2,818,46.654108,-93.417845
2,27,001,770100,3,613,46.573319,-93.562421
3,27,001,770200,1,798,46.915144,-93.617506
4,27,001,770200,2,665,46.716324,-93.573290


In [28]:
bg["fips"] = (
    bg["STATEFP"].str.zfill(2)
    + bg["COUNTYFP"].str.zfill(3)
)

bg["block_group_geoid"] = (
    bg["STATEFP"].str.zfill(2)
    + bg["COUNTYFP"].str.zfill(3)
    + bg["TRACTCE"].str.zfill(6)
    + bg["BLKGRPCE"].str.zfill(1)
)

print(
    "Minnesota counties represented:",
    bg["fips"].nunique()
)

print(
    "Missing latitude:",
    bg["LATITUDE"].isna().sum()
)

print(
    "Missing longitude:",
    bg["LONGITUDE"].isna().sum()
)

print(
    "Total 2020 population:",
    bg["POPULATION"].sum()
)

Minnesota counties represented: 87
Missing latitude: 0
Missing longitude: 0
Total 2020 population: 5706494


In [29]:
located_sites = site_locations[
    site_locations["block_geoid"].notna()
].copy()

located_sites["block_geoid_text"] = (
    pd.to_numeric(
        located_sites["block_geoid"],
        errors="coerce"
    )
    .round()
    .astype("Int64")
    .astype("string")
    .str.zfill(15)
)

print(
    "Located supply sites:",
    len(located_sites)
)

print(
    "Unique block GEOIDs:",
    located_sites[
        "block_geoid_text"
    ].nunique()
)

display(
    located_sites[
        [
            "_id_site",
            "activity_county",
            "block_geoid_text",
            "z12MoAverage_Pounds"
        ]
    ].head()
)

Located supply sites: 473
Unique block GEOIDs: 440


,_id_site,activity_county,block_geoid_text,z12MoAverage_Pounds
0,1066,Hennepin,270030505054009,130862.0
1,1067,Stearns,271450104021049,3787.0
2,1069,Hennepin,270531099001002,29798.0
3,1070,Benton,270090202031035,3550.0
4,1071,Wright,271711004023003,32366.0


In [30]:
acs_2024 = pd.read_csv(
    "../data/public/acs_county.csv",
    dtype={"fips": str}
)

acs_2024 = acs_2024[
    acs_2024["year"] == 2024
][
    [
        "fips",
        "county"
    ]
].copy()

acs_2024 = acs_2024.rename(
    columns={
        "county": "block_county_name"
    }
)

located_sites["block_county_fips"] = (
    located_sites[
        "block_geoid_text"
    ]
    .str[:5]
)

located_sites = located_sites.merge(
    acs_2024,
    left_on="block_county_fips",
    right_on="fips",
    how="left",
    validate="many_to_one"
)

located_sites[
    "county_location_match"
] = (
    located_sites["activity_county"]
    == located_sites["block_county_name"]
)

print(
    "Located sites:",
    len(located_sites)
)

print(
    "County matches:",
    located_sites[
        "county_location_match"
    ].sum()
)

print(
    "County mismatches:",
    (
        ~located_sites[
            "county_location_match"
        ]
    ).sum()
)

print(
    "Missing physical county name:",
    located_sites[
        "block_county_name"
    ].isna().sum()
)

Located sites: 473
County matches: 452
County mismatches: 21
Missing physical county name: 0


In [31]:
county_mismatches = located_sites[
    ~located_sites[
        "county_location_match"
    ]
].copy()

display(
    county_mismatches[
        [
            "_id_site",
            "activity_county",
            "City",
            "Zip",
            "block_county_fips",
            "block_county_name",
            "block_geoid_text",
            "annual_visits"
        ]
    ]
    .sort_values(
        [
            "activity_county",
            "annual_visits"
        ],
        ascending=[
            True,
            False
        ]
    )
)
print(
    "Mismatch share of located sites:",
    100
    * len(county_mismatches)
    / len(located_sites)
)

print(
    "Mismatch share of annual visits:",
    100
    * county_mismatches[
        "annual_visits"
    ].sum()
    / located_sites[
        "annual_visits"
    ].sum()
)

,_id_site,activity_county,City,Zip,block_county_fips,block_county_name,block_geoid_text,annual_visits
324,14473,Anoka,Elko New Market,55054.0,27139,Scott,271390811031006,8086.0
351,14584,Beltrami,Cass Lake,56633.0,27021,Cass,270219400022003,842.0
349,14549,Chisago,Wyoming,55092.0,27003,Anoka,270030516003030,2375.0
0,1066,Hennepin,Anoka,55303.0,27003,Anoka,270030505054009,26353.0
147,1261,Hennepin,Shakopee,55379.0,27139,Scott,271390804001012,13979.0
378,14681,Hennepin,NaN,NaN,27123,Ramsey,271230413022006,7495.0
142,1256,Hennepin,NaN,NaN,27123,Ramsey,271230413022006,1773.0
297,14366,Itasca,S. Lake,56681.0,27013,Blue Earth,270131709002107,630.0
377,14680,Koochiching,Nett Lake,55772.0,27137,St. Louis,271370155031270,2259.0
101,1208,Koochiching,Grand Rapids,55744.0,27061,Itasca,270614808031001,361.0


Mismatch share of located sites: 4.439746300211416
Mismatch share of annual visits: 3.5196653747764377


In [32]:

test_geoid = (
    located_sites["block_geoid_text"]
    .dropna()
    .iloc[0]
)

tiger_url = (
    "https://tigerweb.geo.census.gov/"
    "arcgis/rest/services/"
    "TIGERweb/Tracts_Blocks/"
    "MapServer/2/query"
)

params = {
    "where": f"GEOID = '{test_geoid}'",
    "outFields": (
        "GEOID,CENTLAT,CENTLON,"
        "STATE,COUNTY,TRACT,BLOCK"
    ),
    "returnGeometry": "false",
    "f": "json"
}

query_url = (
    tiger_url
    + "?"
    + urlencode(params)
)

with urlopen(
    query_url,
    timeout=30
) as response:
    test_result = json.loads(
        response.read().decode("utf-8")
    )

print(
    "Test GEOID:",
    test_geoid
)

print(
    "Number of returned features:",
    len(
        test_result.get(
            "features",
            []
        )
    )
)

print(
    test_result.get(
        "features",
        []
    )
)

Test GEOID: 270030505054009
Number of returned features: 1
[{'attributes': {'GEOID': '270030505054009', 'CENTLAT': '+45.2063027', 'CENTLON': '-093.3742607', 'STATE': '27', 'COUNTY': '003', 'TRACT': '050505', 'BLOCK': '4009'}}]


In [33]:
from pathlib import Path

block_coordinate_file = Path(
    "../data/processed/"
    "site_block_coordinates.csv"
)

block_geoids = (
    located_sites[
        "block_geoid_text"
    ]
    .dropna()
    .drop_duplicates()
    .tolist()
)


if block_coordinate_file.exists():

    print(
        "Loading saved block coordinates."
    )

    block_coordinates = pd.read_csv(
        block_coordinate_file,
        dtype={
            "block_geoid_text": str
        }
    )

else:

    print(
        "Saved coordinates not found."
    )

    print(
        "Downloading coordinates "
        "from Census TIGERweb."
    )

    block_coordinates_list = []

    batch_size = 50

    for start in range(
        0,
        len(block_geoids),
        batch_size
    ):

        batch = block_geoids[
            start:start + batch_size
        ]

        quoted_geoids = []

        for geoid in batch:

            quoted_geoids.append(
                f"'{geoid}'"
            )

        where_clause = (
            "GEOID IN ("
            + ",".join(
                quoted_geoids
            )
            + ")"
        )

        params = {
            "where": where_clause,
            "outFields": (
                "GEOID,CENTLAT,CENTLON,"
                "STATE,COUNTY,TRACT,BLOCK"
            ),
            "returnGeometry": "false",
            "f": "json"
        }

        request_data = urlencode(
            params
        ).encode(
            "utf-8"
        )

        request = Request(
            tiger_url,
            data=request_data,
            method="POST"
        )

        with urlopen(
            request,
            timeout=60
        ) as response:

            result = json.loads(
                response.read()
                .decode(
                    "utf-8"
                )
            )

        features = result.get(
            "features",
            []
        )

        for feature in features:

            attributes = feature[
                "attributes"
            ]

            block_coordinates_list.append(
                {
                    "block_geoid_text":
                        attributes[
                            "GEOID"
                        ],
                    "site_latitude":
                        float(
                            attributes[
                                "CENTLAT"
                            ]
                        ),
                    "site_longitude":
                        float(
                            attributes[
                                "CENTLON"
                            ]
                        )
                }
            )

        print(
            "Processed:",
            min(
                start
                + batch_size,
                len(
                    block_geoids
                )
            ),
            "of",
            len(
                block_geoids
            )
        )

    block_coordinates = pd.DataFrame(
        block_coordinates_list
    )

    block_coordinates.to_csv(
        block_coordinate_file,
        index=False
    )

    print(
        "Saved block coordinates:",
        block_coordinate_file
    )

Loading saved block coordinates.


In [34]:
block_coordinates = pd.DataFrame(
    block_coordinates
)

print(
    "Requested unique blocks:",
    len(block_geoids)
)

print(
    "Returned unique blocks:",
    block_coordinates[
        "block_geoid_text"
    ].nunique()
)

print(
    "Duplicate returned blocks:",
    block_coordinates[
        "block_geoid_text"
    ].duplicated().sum()
)

print(
    "Missing latitude:",
    block_coordinates[
        "site_latitude"
    ].isna().sum()
)

print(
    "Missing longitude:",
    block_coordinates[
        "site_longitude"
    ].isna().sum()
)

display(
    block_coordinates.head()
)

Requested unique blocks: 440
Returned unique blocks: 440
Duplicate returned blocks: 0
Missing latitude: 0
Missing longitude: 0


,block_geoid_text,site_latitude,site_longitude
0,271711007042054,45.166961,-93.854673
1,270090202031035,45.752066,-94.218439
2,271051054001022,43.634835,-95.581658
3,270530203042017,45.063064,-93.324820
4,271119612002075,46.281669,-95.716265


In [35]:
# Make this cell safe to rerun.
# Remove coordinate columns from any earlier merge.

coordinate_columns_to_remove = [
    "site_latitude",
    "site_longitude",
    "site_latitude_x",
    "site_latitude_y",
    "site_longitude_x",
    "site_longitude_y"
]

existing_coordinate_columns = []

for column in coordinate_columns_to_remove:

    if column in located_sites.columns:

        existing_coordinate_columns.append(
            column
        )

if len(existing_coordinate_columns) > 0:

    located_sites = located_sites.drop(
        columns=existing_coordinate_columns
    )

    print(
        "Removed old coordinate columns:",
        existing_coordinate_columns
    )


# Merge the saved/downloaded Census coordinates
located_sites = located_sites.merge(
    block_coordinates,
    on="block_geoid_text",
    how="left",
    validate="many_to_one"
)

print(
    "Located sites:",
    len(located_sites)
)

print(
    "Missing coordinates after merge:",
    located_sites[
        "site_latitude"
    ].isna().sum()
)

display(
    located_sites[
        [
            "_id_site",
            "activity_county",
            "block_county_name",
            "site_latitude",
            "site_longitude"
        ]
    ].head()
)

Located sites: 473
Missing coordinates after merge: 0


,_id_site,activity_county,block_county_name,site_latitude,site_longitude
0,1066,Hennepin,Anoka,45.206303,-93.374261
1,1067,Stearns,Stearns,45.633935,-94.572849
2,1069,Hennepin,Hennepin,44.933212,-93.278839
3,1070,Benton,Benton,45.752066,-94.218439
4,1071,Wright,Wright,45.259764,-94.100704


In [36]:
serving_sites = located_sites[
    located_sites["annual_visits"] > 0
].copy()

serving_sites[
    "supply_equal"
] = 1.0

serving_sites[
    "supply_pounds"
] = serving_sites[
    "z12MoAverage_Pounds"
]

print(
    "Located sites:",
    len(located_sites)
)

print(
    "Located sites with positive visits:",
    len(serving_sites)
)

print(
    "Zero-visit located sites excluded:",
    (
        located_sites[
            "annual_visits"
        ] == 0
    ).sum()
)

Located sites: 473
Located sites with positive visits: 467
Zero-visit located sites excluded: 6


In [37]:
acs_demand = pd.read_csv(
    "../data/public/acs_county.csv",
    dtype={"fips": str}
)

acs_2024_demand = acs_demand[
    acs_demand["year"] == 2024
][
    [
        "fips",
        "county",
        "population"
    ]
].copy()

mmg = pd.read_excel(
    "../data/FeedingAmericaData/"
    "MMG2026_2024_Data_To_Share.xlsx",
    sheet_name="County"
)

mmg_2024 = mmg[
    (mmg["State"] == "MN")
    & (mmg["Year"] == 2024)
].copy()

mmg_2024["fips"] = (
    pd.to_numeric(
        mmg_2024["FIPS"],
        errors="coerce"
    )
    .astype("Int64")
    .astype("string")
    .str.zfill(5)
)

mmg_2024["food_insecure_people"] = (
    mmg_2024[
        "# of Food Insecure Persons Overall"
    ]
)

demand_county = acs_2024_demand.merge(
    mmg_2024[
        [
            "fips",
            "food_insecure_people"
        ]
    ],
    on="fips",
    how="left",
    validate="one_to_one"
)

demand_county["food_insecurity_rate"] = (
    demand_county["food_insecure_people"]
    / demand_county["population"]
)

demand_county["benchmark_use_rate"] = (
    0.08
    + 0.34
    * demand_county["food_insecurity_rate"]
)

demand_county["demand_unscaled"] = (
    demand_county["population"]
    * demand_county["benchmark_use_rate"]
)

print(
    "Demand counties:",
    len(demand_county)
)

print(
    "Missing food-insecurity values:",
    demand_county[
        "food_insecurity_rate"
    ].isna().sum()
)

display(
    demand_county[
        demand_county["county"]
        == "Wilkin"
    ]
)

Demand counties: 87
Missing food-insecurity values: 0


,fips,county,population,food_insecure_people,food_insecurity_rate,benchmark_use_rate,demand_unscaled
83,27167,Wilkin,6371,810.0,0.127139,0.123227,785.08


In [38]:
bg["county_bg_population"] = (
    bg.groupby("fips")[
        "POPULATION"
    ]
    .transform("sum")
)

bg["county_population_share"] = (
    bg["POPULATION"]
    / bg["county_bg_population"]
)

print(
    "Counties:",
    bg["fips"].nunique()
)

print(
    "County shares sum to:"
)

display(
    bg.groupby("fips")[
        "county_population_share"
    ]
    .sum()
    .describe()
)

Counties: 87
County shares sum to:


count    87.0
mean      1.0
std       0.0
min       1.0
25%       1.0
50%       1.0
75%       1.0
max       1.0
Name: county_population_share, dtype: float64

In [39]:
demand_points = bg.merge(
    demand_county[
        [
            "fips",
            "county",
            "population",
            "food_insecurity_rate",
            "benchmark_use_rate",
            "demand_unscaled"
        ]
    ],
    on="fips",
    how="left",
    validate="many_to_one"
)

demand_points["demand"] = (
    demand_points["demand_unscaled"]
    * demand_points[
        "county_population_share"
    ]
)

print(
    "Demand points:",
    len(demand_points)
)

print(
    "Counties:",
    demand_points[
        "fips"
    ].nunique()
)

print(
    "Missing demand:",
    demand_points[
        "demand"
    ].isna().sum()
)

print(
    "Zero-demand block groups:",
    (
        demand_points[
            "demand"
        ] == 0
    ).sum()
)

print(
    "Total distributed demand:",
    demand_points[
        "demand"
    ].sum()
)

print(
    "Original county demand:",
    demand_county[
        "demand_unscaled"
    ].sum()
)

Demand points: 4706
Counties: 87
Missing demand: 0
Zero-demand block groups: 38
Total distributed demand: 675603.0
Original county demand: 675602.9999999999


In [40]:
demand_check = (
    demand_points
    .groupby(
        [
            "fips",
            "county"
        ],
        as_index=False
    )
    .agg(
        distributed_demand=(
            "demand",
            "sum"
        )
    )
)

demand_check = demand_check.merge(
    demand_county[
        [
            "fips",
            "demand_unscaled"
        ]
    ],
    on="fips",
    how="left",
    validate="one_to_one"
)

demand_check["difference"] = (
    demand_check[
        "distributed_demand"
    ]
    - demand_check[
        "demand_unscaled"
    ]
)

print(
    "Largest absolute difference:",
    demand_check[
        "difference"
    ].abs().max()
)

display(
    demand_check[
        demand_check["county"]
        .isin(
            [
                "Pope",
                "Lac qui Parle",
                "Wilkin"
            ]
        )
    ]
)

Largest absolute difference: 0.0


,fips,county,distributed_demand,demand_unscaled,difference
36,27073,Lac qui Parle,789.56,789.56,0.0
60,27121,Pope,1348.64,1348.64,0.0
83,27167,Wilkin,785.08,785.08,0.0


In [41]:
demand_lat = np.radians(
    demand_points[
        "LATITUDE"
    ].to_numpy()
)

demand_lon = np.radians(
    demand_points[
        "LONGITUDE"
    ].to_numpy()
)

site_lat = np.radians(
    serving_sites[
        "site_latitude"
    ].to_numpy()
)

site_lon = np.radians(
    serving_sites[
        "site_longitude"
    ].to_numpy()
)

lat_difference = (
    site_lat[None, :]
    - demand_lat[:, None]
)

lon_difference = (
    site_lon[None, :]
    - demand_lon[:, None]
)

haversine_a = (
    np.sin(
        lat_difference / 2
    ) ** 2
    +
    np.cos(
        demand_lat[:, None]
    )
    * np.cos(
        site_lat[None, :]
    )
    * np.sin(
        lon_difference / 2
    ) ** 2
)

central_angle = (
    2
    * np.arcsin(
        np.sqrt(
            haversine_a
        )
    )
)

earth_radius_miles = 3958.7613

distance_miles = (
    earth_radius_miles
    * central_angle
)

print(
    "Distance matrix shape:",
    distance_miles.shape
)

print(
    "Minimum distance:",
    distance_miles.min()
)

print(
    "Maximum distance:",
    distance_miles.max()
)

Distance matrix shape: (4706, 467)
Minimum distance: 0.005405096125270911
Maximum distance: 447.14770717446066


In [42]:
demand_points[
    "nearest_site_miles"
] = (
    distance_miles.min(
        axis=1
    )
)

print(
    demand_points[
        "nearest_site_miles"
    ]
    .describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

print(
    "\nPopulation-weighted / demand-weighted summary:"
)

for threshold in [
    5,
    10,
    15,
    20,
    30,
    45
]:

    demand_within = (
        demand_points.loc[
            demand_points[
                "nearest_site_miles"
            ] <= threshold,
            "demand"
        ].sum()
        /
        demand_points[
            "demand"
        ].sum()
    )

    print(
        threshold,
        "miles:",
        round(
            100 * demand_within,
            2
        ),
        "%"
    )

count    4706.000000
mean        2.922943
std         3.473469
min         0.005405
50%         1.644988
75%         3.666502
90%         7.189172
95%        10.118083
99%        15.693291
max        40.044428
Name: nearest_site_miles, dtype: float64

Population-weighted / demand-weighted summary:
5 miles: 83.83 %
10 miles: 95.95 %
15 miles: 98.98 %
20 miles: 99.67 %
30 miles: 99.96 %
45 miles: 100.0 %


In [43]:
print(
    "Demand points:",
    len(demand_points)
)

print(
    "Counties:",
    demand_points["fips"].nunique()
)

print(
    "Missing demand:",
    demand_points["demand"].isna().sum()
)

print(
    "Distributed demand total:",
    demand_points["demand"].sum()
)

print(
    "County demand total:",
    demand_county["demand_unscaled"].sum()
)

print(
    "Difference:",
    demand_points["demand"].sum()
    - demand_county["demand_unscaled"].sum()
)

Demand points: 4706
Counties: 87
Missing demand: 0
Distributed demand total: 675603.0
County demand total: 675602.9999999999
Difference: 1.1641532182693481e-10


In [44]:
weights = np.zeros_like(
    distance_miles
)

weights[
    distance_miles <= 5
] = 1.00

weights[
    (distance_miles > 5)
    & (distance_miles <= 10)
] = 0.75

weights[
    (distance_miles > 10)
    & (distance_miles <= 20)
] = 0.50

weights[
    (distance_miles > 20)
    & (distance_miles <= 30)
] = 0.25

print(
    "Weight matrix shape:",
    weights.shape
)

print(
    "Nonzero demand-site connections:",
    np.count_nonzero(weights)
)

print(
    "Share of all possible connections:",
    100
    * np.count_nonzero(weights)
    / weights.size
)

Weight matrix shape: (4706, 467)
Nonzero demand-site connections: 495349
Share of all possible connections: 22.539407071568395


In [45]:
demand_array = (
    demand_points["demand"]
    .to_numpy()
)

competing_demand = (
    weights
    * demand_array[:, None]
).sum(
    axis=0
)

serving_sites = (
    serving_sites
    .reset_index(drop=True)
    .copy()
)

serving_sites[
    "competing_demand"
] = competing_demand

serving_sites[
    "site_supply_ratio"
] = (
    serving_sites["supply_equal"]
    / serving_sites["competing_demand"]
)

print(
    "Sites:",
    len(serving_sites)
)

print(
    "Sites with zero nearby demand:",
    (
        serving_sites[
            "competing_demand"
        ] == 0
    ).sum()
)

print(
    serving_sites[
        "competing_demand"
    ].describe()
)

Sites: 467
Sites with zero nearby demand: 0
count       467.000000
mean      81068.699505
std       92855.276473
min          92.678929
25%        3824.079324
50%       14939.127300
75%      188821.052012
max      227258.558594
Name: competing_demand, dtype: float64


In [46]:
site_supply_ratio = (
    serving_sites[
        "site_supply_ratio"
    ].to_numpy()
)

access_equal = (
    weights
    @ site_supply_ratio
)

demand_points[
    "access_equal"
] = access_equal

print(
    demand_points[
        "access_equal"
    ].describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

count    4706.000000
mean        0.000717
std         0.000433
min         0.000000
10%         0.000426
25%         0.000503
50%         0.000600
75%         0.000714
90%         0.001180
95%         0.001546
99%         0.002307
max         0.007810
Name: access_equal, dtype: float64


In [47]:
demand_points[
    "access_times_demand"
] = (
    demand_points[
        "access_equal"
    ]
    * demand_points[
        "demand"
    ]
)

total_accessible_supply = (
    demand_points[
        "access_times_demand"
    ].sum()
)

total_site_supply = (
    serving_sites[
        "supply_equal"
    ].sum()
)

print(
    "Recovered supply through 2SFCA:",
    total_accessible_supply
)

print(
    "Original site supply:",
    total_site_supply
)

print(
    "Difference:",
    total_accessible_supply
    - total_site_supply
)

Recovered supply through 2SFCA: 467.0
Original site supply: 467.0
Difference: 0.0


In [48]:
county_access = (
    demand_points
    .groupby(
        [
            "fips",
            "county"
        ],
        as_index=False
    )
    .agg(
        total_demand=(
            "demand",
            "sum"
        ),
        weighted_access_sum=(
            "access_times_demand",
            "sum"
        )
    )
)

county_access[
    "access_2sfca"
] = (
    county_access[
        "weighted_access_sum"
    ]
    / county_access[
        "total_demand"
    ]
)

statewide_access = (
    county_access[
        "weighted_access_sum"
    ].sum()
    / county_access[
        "total_demand"
    ].sum()
)

county_access[
    "access_index"
] = (
    county_access[
        "access_2sfca"
    ]
    / statewide_access
)

county_access[
    "access_rank"
] = (
    county_access[
        "access_index"
    ]
    .rank(
        method="min",
        ascending=True
    )
    .astype(int)
)

print(
    "Statewide average accessibility:",
    statewide_access
)

print(
    "Counties:",
    len(county_access)
)

display(
    county_access
    .sort_values(
        "access_rank"
    )
    .head(15)
)

Statewide average accessibility: 0.0006912343491665963
Counties: 87


,fips,county,total_demand,weighted_access_sum,access_2sfca,access_index,access_rank
12,27025,Chisago,6417.48,2.122790,0.000331,0.478539,1
85,27171,Wright,15849.72,5.292227,0.000334,0.483049,2
70,27141,Sherburne,10890.60,3.858932,0.000354,0.512613,3
42,27085,McLeod,4443.24,1.747032,0.000393,0.568821,4
83,27167,Wilkin,785.08,0.312013,0.000397,0.574955,5
37,27075,Lake,1302.12,0.536566,0.000412,0.596138,6
81,27163,Washington,29742.24,12.911484,0.000434,0.628025,7
46,27093,Meeker,2819.24,1.276061,0.000453,0.654808,8
9,27019,Carver,11717.08,5.404629,0.000461,0.667300,9
29,27059,Isanti,4937.68,2.288684,0.000464,0.670560,10


In [49]:
spatial_targets = [
    "Lac qui Parle",
    "Pope",
    "Norman",
    "Nicollet",
    "Scott",
    "Wilkin"
]

display(
    county_access[
        county_access[
            "county"
        ].isin(
            spatial_targets
        )
    ][
        [
            "county",
            "total_demand",
            "access_2sfca",
            "access_index",
            "access_rank"
        ]
    ]
    .sort_values(
        "access_rank"
    )
)

,county,total_demand,access_2sfca,access_index,access_rank
83,Wilkin,785.08,0.000397,0.574955,5
69,Scott,16118.16,0.000484,0.699715,11
51,Nicollet,4058.48,0.000608,0.879900,24
60,Pope,1348.64,0.000787,1.139184,38
36,Lac qui Parle,789.56,0.001239,1.792971,63
53,Norman,771.16,0.001829,2.646071,79


In [50]:
zero_access_points = demand_points[
    demand_points["access_equal"] == 0
].copy()

print(
    "Zero-access block groups:",
    len(zero_access_points)
)

print(
    "Share of statewide demand with zero access:",
    100
    * zero_access_points["demand"].sum()
    / demand_points["demand"].sum()
)

display(
    zero_access_points
    .groupby(
        "county",
        as_index=False
    )
    .agg(
        zero_access_demand=(
            "demand",
            "sum"
        )
    )
    .sort_values(
        "zero_access_demand",
        ascending=False
    )
    .head(10)
)

Zero-access block groups: 2
Share of statewide demand with zero access: 0.03580667287304581


,county,zero_access_demand
0,Beltrami,241.910956


In [51]:
print(
    "Serving sites:",
    len(serving_sites)
)

print(
    "Missing 12-month average pounds:",
    serving_sites[
        "z12MoAverage_Pounds"
    ].isna().sum()
)

print(
    "Zero 12-month average pounds:",
    (
        serving_sites[
            "z12MoAverage_Pounds"
        ] == 0
    ).sum()
)

display(
    serving_sites[
        serving_sites[
            "z12MoAverage_Pounds"
        ].isna()
        |
        (
            serving_sites[
                "z12MoAverage_Pounds"
            ] == 0
        )
    ][
        [
            "_id_site",
            "activity_county",
            "annual_visits",
            "annual_pounds",
            "z12MoAverage_Pounds"
        ]
    ]
)

Serving sites: 467
Missing 12-month average pounds: 0
Zero 12-month average pounds: 4


,_id_site,activity_county,annual_visits,annual_pounds,z12MoAverage_Pounds
319,14470,Ramsey,317.0,21562.0,0.0
329,14487,Hennepin,546.0,38220.0,0.0
342,14512,Hennepin,173.0,3938.0,0.0
445,15353,Hennepin,430.0,26884.0,0.0


In [52]:
valid_pounds = (
    serving_sites[
        "z12MoAverage_Pounds"
    ].notna()
)

pounds_sites = serving_sites[
    valid_pounds
].copy()

weights_pounds = weights[
    :,
    valid_pounds.to_numpy()
]

pounds_competing_demand = (
    weights_pounds
    * demand_array[:, None]
).sum(
    axis=0
)

pounds_supply = (
    pounds_sites[
        "z12MoAverage_Pounds"
    ].to_numpy()
)

pounds_site_ratio = (
    pounds_supply
    / pounds_competing_demand
)

access_pounds = (
    weights_pounds
    @ pounds_site_ratio
)

demand_points[
    "access_pounds"
] = access_pounds

print(
    "Sites in pounds sensitivity:",
    len(pounds_sites)
)

print(
    demand_points[
        "access_pounds"
    ].describe()
)

Sites in pounds sensitivity: 467
count    4706.000000
mean       19.531634
std         7.549709
min         0.000000
25%        13.194297
50%        20.441824
75%        25.891126
max        97.537654
Name: access_pounds, dtype: float64


In [53]:
demand_points[
    "access_pounds_times_demand"
] = (
    demand_points[
        "access_pounds"
    ]
    * demand_points[
        "demand"
    ]
)

print(
    "Recovered pounds supply:",
    demand_points[
        "access_pounds_times_demand"
    ].sum()
)

print(
    "Original pounds supply:",
    pounds_supply.sum()
)

print(
    "Difference:",
    demand_points[
        "access_pounds_times_demand"
    ].sum()
    - pounds_supply.sum()
)

county_access_pounds = (
    demand_points
    .groupby(
        [
            "fips",
            "county"
        ],
        as_index=False
    )
    .agg(
        total_demand=(
            "demand",
            "sum"
        ),
        weighted_pounds_access=(
            "access_pounds_times_demand",
            "sum"
        )
    )
)

county_access_pounds[
    "access_2sfca_pounds"
] = (
    county_access_pounds[
        "weighted_pounds_access"
    ]
    / county_access_pounds[
        "total_demand"
    ]
)

statewide_pounds_access = (
    county_access_pounds[
        "weighted_pounds_access"
    ].sum()
    / county_access_pounds[
        "total_demand"
    ].sum()
)

county_access_pounds[
    "pounds_access_index"
] = (
    county_access_pounds[
        "access_2sfca_pounds"
    ]
    / statewide_pounds_access
)

county_access_pounds[
    "pounds_access_rank"
] = (
    county_access_pounds[
        "pounds_access_index"
    ]
    .rank(
        method="min",
        ascending=True
    )
    .astype(int)
)

display(
    county_access_pounds
    .sort_values(
        "pounds_access_rank"
    )
    .head(15)
)

Recovered pounds supply: 13162967.0
Original pounds supply: 13162967.0
Difference: 0.0


,fips,county,total_demand,weighted_pounds_access,access_2sfca_pounds,pounds_access_index,pounds_access_rank
37,27075,Lake,1302.12,3889.804784,2.987286,0.153326,1
83,27167,Wilkin,785.08,2661.664936,3.390310,0.174011,2
36,27073,Lac qui Parle,789.56,4576.224248,5.795917,0.297481,3
74,27149,Stevens,1210.92,7159.746942,5.912651,0.303473,4
53,27107,Norman,771.16,4906.962491,6.363093,0.326592,5
75,27151,Swift,1300.04,8303.928325,6.387441,0.327842,6
67,27135,Roseau,1836.16,12140.261914,6.611767,0.339356,7
77,27155,Traverse,405.64,2889.748350,7.123924,0.365643,8
12,27025,Chisago,6417.48,48028.539114,7.484019,0.384125,9
40,27081,Lincoln,679.44,5405.954100,7.956485,0.408375,10


In [54]:
spatial_compare = county_access[
    [
        "fips",
        "county",
        "access_index",
        "access_rank"
    ]
].merge(
    county_access_pounds[
        [
            "fips",
            "pounds_access_index",
            "pounds_access_rank"
        ]
    ],
    on="fips",
    how="left",
    validate="one_to_one"
)

spatial_compare[
    "rank_change"
] = (
    spatial_compare[
        "access_rank"
    ]
    - spatial_compare[
        "pounds_access_rank"
    ]
).abs()

print(
    "Rank correlation:",
    spatial_compare[
        [
            "access_rank",
            "pounds_access_rank"
        ]
    ]
    .corr(
        method="spearman"
    )
    .iloc[0, 1]
)

display(
    spatial_compare
    .sort_values(
        "rank_change",
        ascending=False
    )
    .head(15)
)

Rank correlation: -0.03085587227527885


,fips,county,access_index,access_rank,pounds_access_index,pounds_access_rank,rank_change
77,27155,Traverse,3.649350,84,0.365643,8,76
53,27107,Norman,2.646071,79,0.326592,5,74
81,27163,Washington,0.628025,7,0.950424,73,66
26,27053,Hennepin,0.864853,22,1.372542,87,65
1,27003,Anoka,0.732311,15,1.113211,79,64
61,27123,Ramsey,0.864956,23,1.350287,86,63
18,27037,Dakota,0.720586,14,1.089661,76,62
44,27089,Marshall,2.766422,80,0.473952,19,61
36,27073,Lac qui Parle,1.792971,63,0.297481,3,60
65,27131,Rice,0.710163,12,0.927802,71,59


In [55]:
display(
    spatial_compare[
        spatial_compare[
            "county"
        ].isin(
            [
                "Lac qui Parle",
                "Pope",
                "Norman",
                "Nicollet",
                "Scott",
                "Wilkin"
            ]
        )
    ][
        [
            "county",
            "access_index",
            "access_rank",
            "pounds_access_index",
            "pounds_access_rank",
            "rank_change"
        ]
    ]
    .sort_values(
        "pounds_access_rank"
    )
)

,county,access_index,access_rank,pounds_access_index,pounds_access_rank,rank_change
83,Wilkin,0.574955,5,0.174011,2,3
36,Lac qui Parle,1.792971,63,0.297481,3,60
53,Norman,2.646071,79,0.326592,5,74
60,Pope,1.139184,38,0.556357,32,6
69,Scott,0.699715,11,0.907373,69,58
51,Nicollet,0.879900,24,0.957302,74,50


In [56]:
serving_sites["monthly_pounds_2024"] = (
    serving_sites["annual_pounds"]
    / 12
)

print(
    "Zero actual 2024 pounds:",
    (
        serving_sites[
            "monthly_pounds_2024"
        ] == 0
    ).sum()
)

print(
    "\nPearson correlation:"
)

print(
    serving_sites[
        [
            "z12MoAverage_Pounds",
            "monthly_pounds_2024"
        ]
    ]
    .corr(
        method="pearson"
    )
)

print(
    "\nSpearman correlation:"
)

print(
    serving_sites[
        [
            "z12MoAverage_Pounds",
            "monthly_pounds_2024"
        ]
    ]
    .corr(
        method="spearman"
    )
)

Zero actual 2024 pounds: 0

Pearson correlation:
                     z12MoAverage_Pounds  monthly_pounds_2024
z12MoAverage_Pounds             1.000000             0.945203
monthly_pounds_2024             0.945203             1.000000

Spearman correlation:
                     z12MoAverage_Pounds  monthly_pounds_2024
z12MoAverage_Pounds              1.00000              0.95253
monthly_pounds_2024              0.95253              1.00000


In [57]:
pounds_compare = serving_sites[
    (
        serving_sites[
            "z12MoAverage_Pounds"
        ] > 0
    )
    &
    (
        serving_sites[
            "monthly_pounds_2024"
        ] > 0
    )
].copy()

pounds_compare[
    "pounds_ratio_org_to_2024"
] = (
    pounds_compare[
        "z12MoAverage_Pounds"
    ]
    / pounds_compare[
        "monthly_pounds_2024"
    ]
)

print(
    pounds_compare[
        "pounds_ratio_org_to_2024"
    ].describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

pounds_compare[
    "distance_from_one"
] = abs(
    np.log(
        pounds_compare[
            "pounds_ratio_org_to_2024"
        ]
    )
)

print(
    "\nLargest disagreements:"
)

display(
    pounds_compare[
        [
            "_id_site",
            "activity_county",
            "z12MoAverage_Pounds",
            "monthly_pounds_2024",
            "pounds_ratio_org_to_2024",
            "distance_from_one"
        ]
    ]
    .sort_values(
        "distance_from_one",
        ascending=False
    )
    .head(15)
)

count    463.000000
mean       1.317038
std        2.075486
min        0.186267
10%        0.691917
25%        0.868577
50%        1.031489
75%        1.265453
90%        1.660116
95%        2.345627
99%        6.750686
max       37.097691
Name: pounds_ratio_org_to_2024, dtype: float64

Largest disagreements:


,_id_site,activity_county,z12MoAverage_Pounds,monthly_pounds_2024,pounds_ratio_org_to_2024,distance_from_one
431,15301,Ramsey,3481.0,93.833333,37.097691,3.613555
447,15372,Hennepin,18947.0,1010.750000,18.745486,2.930953
462,15771,St. Louis,4362.0,363.500000,12.000000,2.484907
301,14416,Cass,4157.0,408.833333,10.167958,2.319241
234,14032,Hennepin,2321.0,300.083333,7.734518,2.045693
450,15377,Clay,333.0,54.166667,6.147692,1.816077
463,15825,Hennepin,12130.0,2203.666667,5.504462,1.705559
354,14598,Ramsey,760.0,4080.166667,0.186267,1.680575
401,14772,Hennepin,496.0,2373.833333,0.208945,1.565685
361,14632,Ramsey,503.0,2331.083333,0.215780,1.533498


In [58]:
actual_pounds_supply = (
    serving_sites[
        "annual_pounds"
    ].to_numpy()
)

actual_pounds_site_ratio = (
    actual_pounds_supply
    / competing_demand
)

access_actual_pounds = (
    weights
    @ actual_pounds_site_ratio
)

demand_points[
    "access_actual_pounds"
] = access_actual_pounds

demand_points[
    "actual_pounds_access_times_demand"
] = (
    demand_points[
        "access_actual_pounds"
    ]
    * demand_points[
        "demand"
    ]
)

print(
    "Recovered actual pounds:",
    demand_points[
        "actual_pounds_access_times_demand"
    ].sum()
)

print(
    "Original actual pounds:",
    actual_pounds_supply.sum()
)

Recovered actual pounds: 154098830.41999996
Original actual pounds: 154098830.42


In [59]:
county_actual_pounds = (
    demand_points
    .groupby(
        [
            "fips",
            "county"
        ],
        as_index=False
    )
    .agg(
        total_demand=(
            "demand",
            "sum"
        ),
        weighted_actual_pounds=(
            "actual_pounds_access_times_demand",
            "sum"
        )
    )
)

county_actual_pounds[
    "actual_pounds_access"
] = (
    county_actual_pounds[
        "weighted_actual_pounds"
    ]
    / county_actual_pounds[
        "total_demand"
    ]
)

state_actual_pounds_access = (
    county_actual_pounds[
        "weighted_actual_pounds"
    ].sum()
    / county_actual_pounds[
        "total_demand"
    ].sum()
)

county_actual_pounds[
    "actual_pounds_index"
] = (
    county_actual_pounds[
        "actual_pounds_access"
    ]
    / state_actual_pounds_access
)

county_actual_pounds[
    "actual_pounds_rank"
] = (
    county_actual_pounds[
        "actual_pounds_index"
    ]
    .rank(
        method="min",
        ascending=True
    )
    .astype(int)
)

In [60]:
spatial_three_way = (
    spatial_compare
    .merge(
        county_actual_pounds[
            [
                "fips",
                "actual_pounds_index",
                "actual_pounds_rank"
            ]
        ],
        on="fips",
        how="left",
        validate="one_to_one"
    )
)

print(
    "Organization pounds vs actual 2024 pounds rank correlation:"
)

print(
    spatial_three_way[
        [
            "pounds_access_rank",
            "actual_pounds_rank"
        ]
    ]
    .corr(
        method="spearman"
    )
    .iloc[0, 1]
)

display(
    spatial_three_way[
        spatial_three_way[
            "county"
        ].isin(
            [
                "Lac qui Parle",
                "Pope",
                "Norman",
                "Nicollet",
                "Scott",
                "Wilkin"
            ]
        )
    ][
        [
            "county",
            "access_rank",
            "pounds_access_rank",
            "actual_pounds_rank",
            "access_index",
            "pounds_access_index",
            "actual_pounds_index"
        ]
    ]
)

Organization pounds vs actual 2024 pounds rank correlation:
0.9193154479842531


,county,access_rank,pounds_access_rank,actual_pounds_rank,access_index,pounds_access_index,actual_pounds_index
36,Lac qui Parle,63,3,6,1.792971,0.297481,0.302041
51,Nicollet,24,74,78,0.879900,0.957302,1.089451
53,Norman,79,5,4,2.646071,0.326592,0.241331
60,Pope,38,32,24,1.139184,0.556357,0.500686
69,Scott,11,69,66,0.699715,0.907373,0.878123
83,Wilkin,5,2,1,0.574955,0.174011,0.197339


In [61]:
#reusable 2SFCA calculation
def calculate_2sfca(weight_matrix, supply_array):

    nearby_demand = (
        weight_matrix
        * demand_array[:, None]
    ).sum(axis=0)

    site_ratio = np.zeros(
        len(supply_array),
        dtype=float
    )

    usable_sites = (
        nearby_demand > 0
    )

    site_ratio[
        usable_sites
    ] = (
        supply_array[
            usable_sites
        ]
        / nearby_demand[
            usable_sites
        ]
    )

    point_access = (
        weight_matrix
        @ site_ratio
    )

    weighted_access = (
        point_access
        * demand_array
    )

    result = pd.DataFrame(
        {
            "fips":
                demand_points["fips"],
            "county":
                demand_points["county"],
            "demand":
                demand_array,
            "weighted_access":
                weighted_access
        }
    )

    result = (
        result
        .groupby(
            [
                "fips",
                "county"
            ],
            as_index=False
        )
        .agg(
            total_demand=(
                "demand",
                "sum"
            ),
            weighted_access=(
                "weighted_access",
                "sum"
            )
        )
    )

    result["access"] = (
        result["weighted_access"]
        / result["total_demand"]
    )

    state_access = (
        result[
            "weighted_access"
        ].sum()
        / result[
            "total_demand"
        ].sum()
    )

    result["index"] = (
        result["access"]
        / state_access
    )

    result["rank"] = (
        result["index"]
        .rank(
            method="min",
            ascending=True
        )
        .astype(int)
    )

    return result

In [62]:
#alternative catchments
weights_20 = (
    distance_miles <= 20
).astype(float)

weights_30 = (
    distance_miles <= 30
).astype(float)

weights_45 = (
    distance_miles <= 45
).astype(float)

equal_supply = np.ones(
    len(serving_sites)
)

org_pounds_supply = (
    serving_sites[
        "z12MoAverage_Pounds"
    ]
    .to_numpy()
)

equal_decay = calculate_2sfca(
    weights,
    equal_supply
)

equal_20 = calculate_2sfca(
    weights_20,
    equal_supply
)

equal_30 = calculate_2sfca(
    weights_30,
    equal_supply
)

equal_45 = calculate_2sfca(
    weights_45,
    equal_supply
)

pounds_decay = calculate_2sfca(
    weights,
    org_pounds_supply
)

pounds_20 = calculate_2sfca(
    weights_20,
    org_pounds_supply
)

pounds_30 = calculate_2sfca(
    weights_30,
    org_pounds_supply
)

pounds_45 = calculate_2sfca(
    weights_45,
    org_pounds_supply
)

In [63]:
#test statewide rank stability
sensitivity = equal_decay[
    [
        "fips",
        "county",
        "rank"
    ]
].rename(
    columns={
        "rank":
            "equal_decay_rank"
    }
)

sensitivity = sensitivity.merge(
    equal_20[
        ["fips", "rank"]
    ].rename(
        columns={
            "rank":
                "equal_20_rank"
        }
    ),
    on="fips"
)

sensitivity = sensitivity.merge(
    equal_30[
        ["fips", "rank"]
    ].rename(
        columns={
            "rank":
                "equal_30_rank"
        }
    ),
    on="fips"
)

sensitivity = sensitivity.merge(
    equal_45[
        ["fips", "rank"]
    ].rename(
        columns={
            "rank":
                "equal_45_rank"
        }
    ),
    on="fips"
)

sensitivity = sensitivity.merge(
    pounds_decay[
        ["fips", "rank"]
    ].rename(
        columns={
            "rank":
                "pounds_decay_rank"
        }
    ),
    on="fips"
)

sensitivity = sensitivity.merge(
    pounds_20[
        ["fips", "rank"]
    ].rename(
        columns={
            "rank":
                "pounds_20_rank"
        }
    ),
    on="fips"
)

sensitivity = sensitivity.merge(
    pounds_30[
        ["fips", "rank"]
    ].rename(
        columns={
            "rank":
                "pounds_30_rank"
        }
    ),
    on="fips"
)

sensitivity = sensitivity.merge(
    pounds_45[
        ["fips", "rank"]
    ].rename(
        columns={
            "rank":
                "pounds_45_rank"
        }
    ),
    on="fips"
)

print(
    "Equal-site rank correlations:"
)

display(
    sensitivity[
        [
            "equal_decay_rank",
            "equal_20_rank",
            "equal_30_rank",
            "equal_45_rank"
        ]
    ].corr(
        method="spearman"
    )
)

print(
    "\nPounds-weighted rank correlations:"
)

display(
    sensitivity[
        [
            "pounds_decay_rank",
            "pounds_20_rank",
            "pounds_30_rank",
            "pounds_45_rank"
        ]
    ].corr(
        method="spearman"
    )
)

Equal-site rank correlations:


,equal_decay_rank,equal_20_rank,equal_30_rank,equal_45_rank
equal_decay_rank,1.000000,0.948148,0.974849,0.782551
equal_20_rank,0.948148,1.000000,0.901345,0.675840
equal_30_rank,0.974849,0.901345,1.000000,0.834822
equal_45_rank,0.782551,0.675840,0.834822,1.000000



Pounds-weighted rank correlations:


,pounds_decay_rank,pounds_20_rank,pounds_30_rank,pounds_45_rank
pounds_decay_rank,1.000000,0.939236,0.872822,0.531111
pounds_20_rank,0.939236,1.000000,0.764398,0.424072
pounds_30_rank,0.872822,0.764398,1.000000,0.705548
pounds_45_rank,0.531111,0.424072,0.705548,1.000000


In [64]:
#sensitivity for our six counties
targets = [
    "Lac qui Parle",
    "Pope",
    "Norman",
    "Nicollet",
    "Scott",
    "Wilkin"
]

display(
    sensitivity[
        sensitivity[
            "county"
        ].isin(targets)
    ][
        [
            "county",
            "equal_decay_rank",
            "equal_20_rank",
            "equal_30_rank",
            "equal_45_rank",
            "pounds_decay_rank",
            "pounds_20_rank",
            "pounds_30_rank",
            "pounds_45_rank"
        ]
    ]
)

,county,equal_decay_rank,equal_20_rank,equal_30_rank,equal_45_rank,pounds_decay_rank,pounds_20_rank,pounds_30_rank,pounds_45_rank
36,Lac qui Parle,63,64,61,36,3,6,3,4
51,Nicollet,24,30,21,15,74,76,65,55
53,Norman,79,85,76,84,5,7,10,65
60,Pope,38,42,39,25,32,45,37,15
69,Scott,11,9,23,23,69,67,79,87
83,Wilkin,5,1,30,69,2,1,6,37


In [65]:
#summarize local robustness
local_robustness = sensitivity[
    [
        "fips",
        "county",
        "equal_decay_rank",
        "equal_20_rank",
        "equal_30_rank",
        "pounds_decay_rank",
        "pounds_20_rank",
        "pounds_30_rank"
    ]
].copy()

local_robustness[
    "equal_min_rank"
] = local_robustness[
    [
        "equal_decay_rank",
        "equal_20_rank",
        "equal_30_rank"
    ]
].min(axis=1)

local_robustness[
    "equal_max_rank"
] = local_robustness[
    [
        "equal_decay_rank",
        "equal_20_rank",
        "equal_30_rank"
    ]
].max(axis=1)

local_robustness[
    "pounds_min_rank"
] = local_robustness[
    [
        "pounds_decay_rank",
        "pounds_20_rank",
        "pounds_30_rank"
    ]
].min(axis=1)

local_robustness[
    "pounds_max_rank"
] = local_robustness[
    [
        "pounds_decay_rank",
        "pounds_20_rank",
        "pounds_30_rank"
    ]
].max(axis=1)

display(
    local_robustness[
        local_robustness["county"].isin(
            [
                "Lac qui Parle",
                "Norman",
                "Pope",
                "Nicollet",
                "Scott",
                "Wilkin"
            ]
        )
    ][
        [
            "county",
            "equal_min_rank",
            "equal_max_rank",
            "pounds_min_rank",
            "pounds_max_rank"
        ]
    ]
)

,county,equal_min_rank,equal_max_rank,pounds_min_rank,pounds_max_rank
36,Lac qui Parle,61,64,3,6
51,Nicollet,21,30,65,76
53,Norman,76,85,5,10
60,Pope,38,42,32,45
69,Scott,9,23,67,79
83,Wilkin,1,30,1,6


In [66]:
#audit out-of-state organizations
organization_border = pd.read_excel(
    "../data/Organization.xlsx"
)

organization_border[
    "block_geoid_text"
] = (
    pd.to_numeric(
        organization_border[
            "block_geoid"
        ],
        errors="coerce"
    )
    .round()
    .astype("Int64")
    .astype("string")
    .str.zfill(15)
)

organization_border[
    "block_state_fips"
] = (
    organization_border[
        "block_geoid_text"
    ]
    .str[:2]
)

neighbor_state_fips = [
    "19",  # Iowa
    "38",  # North Dakota
    "46",  # South Dakota
    "55"   # Wisconsin
]

border_candidates = organization_border[
    organization_border[
        "block_state_fips"
    ].isin(
        neighbor_state_fips
    )
].copy()

print(
    "Organizations physically in neighboring states:",
    len(border_candidates)
)

print(
    "\nBy state FIPS:"
)

print(
    border_candidates[
        "block_state_fips"
    ].value_counts()
)

print(
    "\nDistribution-site status:"
)

print(
    border_candidates[
        "isDistributionSite"
    ].value_counts(
        dropna=False
    )
)

print(
    "\nPositive 12-month pounds:",
    (
        border_candidates[
            "z12MoAverage_Pounds"
        ].fillna(0)
        > 0
    ).sum()
)

Organizations physically in neighboring states: 1

By state FIPS:
block_state_fips
38    1
Name: count, dtype: Int64

Distribution-site status:
isDistributionSite
0.0    1
Name: count, dtype: int64

Positive 12-month pounds: 0


In [67]:
#specifically inspect North Dakota/Fargo-area records
north_dakota_candidates = border_candidates[
    border_candidates[
        "block_state_fips"
    ] == "38"
].copy()

display(
    north_dakota_candidates[
        [
            "__ID",
            "City",
            "State",
            "Zip",
            "County",
            "isDistributionSite",
            "zcountDaysOperations",
            "z12MoAverage_Pounds",
            "block_geoid_text"
        ]
    ]
    .sort_values(
        [
            "City",
            "z12MoAverage_Pounds"
        ],
        ascending=[
            True,
            False
        ]
    )
)

,__ID,City,State,Zip,County,isDistributionSite,zcountDaysOperations,z12MoAverage_Pounds,block_geoid_text
4,7.0,Fargo,ND,58102.0,Clay,0.0,NaN,NaN,380170006021030


In [68]:
#assemble the final spatial county table
spatial_final = spatial_three_way[
    [
        "fips",
        "county",
        "access_index",
        "access_rank",
        "pounds_access_index",
        "pounds_access_rank",
        "actual_pounds_index",
        "actual_pounds_rank"
    ]
].copy()

spatial_final = spatial_final.merge(
    local_robustness[
        [
            "fips",
            "equal_min_rank",
            "equal_max_rank",
            "pounds_min_rank",
            "pounds_max_rank"
        ]
    ],
    on="fips",
    how="left",
    validate="one_to_one"
)

print(
    "Rows:",
    len(spatial_final)
)

print(
    "Counties:",
    spatial_final[
        "county"
    ].nunique()
)

display(
    spatial_final.head()
)

Rows: 87
Counties: 87


,fips,county,access_index,access_rank,pounds_access_index,pounds_access_rank,actual_pounds_index,actual_pounds_rank,equal_min_rank,equal_max_rank,pounds_min_rank,pounds_max_rank
0,27001,Aitkin,2.220071,71,0.476019,22,0.481538,23,70,76,19,22
1,27003,Anoka,0.732311,15,1.113211,79,1.070197,76,10,18,73,82
2,27005,Becker,0.899707,25,0.692319,48,0.708340,51,21,27,33,58
3,27007,Beltrami,1.289900,45,0.786419,60,0.774832,61,36,45,51,60
4,27009,Benton,0.737158,16,0.753167,56,0.708108,50,8,16,32,61


In [71]:
spatial_final[
    "spatial_interpretation"
] = "No special classification"

spatial_final.loc[
    spatial_final["county"]
    == "Lac qui Parle",
    "spatial_interpretation"
] = (
    "Relatively high and robust site-count access; "
    "geographic site availability is unlikely to be "
    "the primary explanation for low utilization."
)

spatial_final.loc[
    spatial_final["county"]
    == "Norman",
    "spatial_interpretation"
] = (
    "Very high site-count access across local "
    "catchment sensitivities; distance to listed "
    "sites does not explain the low utilization."
)

spatial_final.loc[
    spatial_final["county"]
    == "Pope",
    "spatial_interpretation"
] = (
    "Approximately middle-range site-count access; "
    "not an extreme geographic-access deficit."
)

spatial_final.loc[
    spatial_final["county"]
    == "Nicollet",
    "spatial_interpretation"
] = (
    "Somewhat lower site-count access; "
    "regional travel or neighboring-county use "
    "remains a plausible partial explanation."
)

spatial_final.loc[
    spatial_final["county"]
    == "Scott",
    "spatial_interpretation"
] = (
    "Relatively low site-count access, but the "
    "low-utilization anomaly disappears after "
    "the physical-county attribution correction."
)

spatial_final.loc[
    spatial_final["county"]
    == "Wilkin",
    "spatial_interpretation"
] = (
    "Measured Minnesota-network access is low "
    "under shorter catchments but highly sensitive "
    "to catchment assumptions; cross-border access "
    "remains unresolved."
)

display(
    spatial_final[
        spatial_final[
            "county"
        ].isin(
            [
                "Lac qui Parle",
                "Norman",
                "Pope",
                "Scott",
                "Nicollet",
                "Wilkin"
            ]
        )
    ][
        [
            "county",
            "access_rank",
            "equal_min_rank",
            "equal_max_rank",
            "spatial_interpretation"
        ]
    ]
)

,county,access_rank,equal_min_rank,equal_max_rank,spatial_interpretation
36,Lac qui Parle,63,61,64,Relatively high and robust site-count access; ...
51,Nicollet,24,21,30,Somewhat lower site-count access; regional tra...
53,Norman,79,76,85,Very high site-count access across local catch...
60,Pope,38,38,42,Approximately middle-range site-count access; ...
69,Scott,11,9,23,"Relatively low site-count access, but the low-..."
83,Wilkin,5,1,30,Measured Minnesota-network access is low under...


In [72]:
#save the spatial output
spatial_final.to_csv(
    "../data/processed/"
    "spatial_access_2024.csv",
    index=False
)

print(
    "Saved rows:",
    len(spatial_final)
)

Saved rows: 87
